In [0]:
import uuid
from datetime import datetime
from pyspark.sql import functions as F

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.utcnow()
results = []

def log(check, table, column, failed, total):
    results.append((RUN_ID, RUN_TS, check, table, column,
                    int(failed), int(total), "PASS" if failed == 0 else "FAIL"))

def check_not_null(table, cols):
    df = spark.table(table); total = df.count()
    for c in cols:
        log("not_null", table, c, df.filter(F.col(c).isNull()).count(), total)

def check_unique(table, cols):
    df = spark.table(table); total = df.count()
    dupes = total - df.dropDuplicates(cols).count()
    log("unique", table, ",".join(cols), dupes, total)

def check_ref_integrity(child, child_col, parent, parent_col):
    c, p = spark.table(child), spark.table(parent)
    orphans = c.join(p, c[child_col] == p[parent_col], "left_anti").count()
    log("referential_integrity", child, f"{child_col}->{parent}.{parent_col}", orphans, c.count())

In [0]:
%skip
BRONZE_TABLES = ["orders", "order_items", "customers", "products", "sellers", "category_translation"]

def baseline_schema():
    rows = [(t, c) for t in BRONZE_TABLES
            for c in spark.table(f"workspace.bronze.{t}").columns if not c.startswith("_")]
    (spark.createDataFrame(rows, ["table_name", "column_name"])
          .write.mode("overwrite").saveAsTable("workspace.dq.expected_schema"))

baseline_schema()   # run ONCE while your data is known-good, then comment it out

In [0]:
def check_schema_drift():
    expected = spark.table("workspace.dq.expected_schema").collect()
    for t in BRONZE_TABLES:
        exp = {r.column_name for r in expected if r.table_name == t}
        act = {c for c in spark.table(f"workspace.bronze.{t}").columns if not c.startswith("_")}
        drift = len(exp - act) + len(act - exp)   # missing + unexpected columns
        log("schema_drift", f"workspace.bronze.{t}", f"missing={sorted(exp-act)} new={sorted(act-exp)}", drift, len(exp))

In [0]:
check_schema_drift()

check_not_null("workspace.silver.orders", ["order_id", "customer_id", "order_purchase_timestamp"])
check_not_null("workspace.gold.fact_orders", ["order_id", "customer_id", "product_id", "seller_id", "order_date", "price"])

check_unique("workspace.silver.orders", ["order_id"])
check_unique("workspace.gold.fact_orders", ["order_id", "order_item_id"])
check_unique("workspace.gold.dim_customer", ["customer_id"])
check_unique("workspace.gold.dim_product", ["product_id"])
check_unique("workspace.gold.dim_seller", ["seller_id"])

F_ = "workspace.gold.fact_orders"
check_ref_integrity(F_, "customer_id", "workspace.gold.dim_customer", "customer_id")
check_ref_integrity(F_, "product_id", "workspace.gold.dim_product", "product_id")
check_ref_integrity(F_, "seller_id", "workspace.gold.dim_seller", "seller_id")
check_ref_integrity(F_, "order_date", "workspace.gold.dim_date", "date")

cols = ["run_id", "run_timestamp", "check_name", "table_name", "column_name",
        "failed_count", "total_count", "status"]
(spark.createDataFrame(results, cols)
      .write.format("delta").mode("append").saveAsTable("workspace.dq.dq_results"))

display(spark.table("workspace.dq.dq_results").filter(F.col("run_id") == RUN_ID))

failed = [r for r in results if r[7] == "FAIL"]
if failed:
    raise Exception(f"{len(failed)} data quality checks failed in run {RUN_ID}")
